# Recolección y organización de datos TIBAITATA

## Importar datos directamente de la API de datos abiertos colombia

In [ ]:
# -*- coding: utf-8 -*-

import os
import pandas as pd
from pathlib import Path
from sodapy import Socrata
import matplotlib.pyplot as plt



# Crear carpeta para guardar datasets si no existe
os.makedirs("./Datasets", exist_ok=True)

client = Socrata("www.datos.gov.co", None)

results = client.get("hp9r-jxuu", limit=5000)

catalogoEstaciones = pd.DataFrame.from_records(results)

# Normalizar nombres de columnas para evitar problemas con caracteres especiales
catalogoEstaciones.columns = [str(col).strip() for col in catalogoEstaciones.columns]

catalogoEstaciones.head()


,codigo,nombre,categoria,tecnologia,estado,departamento,municipio,ubicaci_n,altitud,longitud,latitud,fecha_instalacion,area_operativa,area_hidrografica,zona_hidrografica,subzona_hidrografica,entidad,corriente,fecha_suspension
0,0024030700,COVARACHIA [24030700],Pluviométrica,Convencional,Activa,Boyacá,Covarachía,"{'latitude': '-72.739228889', 'longitude': '6....",2244,-72.739228889,6.500608056,15/06/1974,Area Operativa 06 - Boyacá-Casanare-Vichada,Magdalena Cauca,Sogamoso,Río Chicamocha,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,NaN,NaN
1,0035075040,INSTITUCION AGRICOLA MACANAL [35075040],Climatológica Principal,Convencional,Activa,Boyacá,Macanal,"{'latitude': '-73.316151944', 'longitude': '4....",1742,-73.316151944,4.974416111,15/07/1982,Area Operativa 06 - Boyacá-Casanare-Vichada,Orinoco,Meta,Río Garagoa,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,NaN,NaN
2,0035070170,NAZARETH [35070170],Pluviométrica,Convencional,Activa,Boyacá,Santa María,"{'latitude': '-73.219595', 'longitude': '4.735...",439,-73.219595,4.735228889,15/09/1972,Area Operativa 06 - Boyacá-Casanare-Vichada,Orinoco,Meta,Río Guavio,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,NaN,NaN
3,0035190030,CHAMEZA [35190030],Pluviográfica,Convencional,Activa,Casanare,Chámeza,"{'latitude': '-72.871813889', 'longitude': '5....",1129,-72.871813889,5.215341944,15/11/1974,Area Operativa 06 - Boyacá-Casanare-Vichada,Orinoco,Meta,Río Cusiana,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,NaN,NaN
4,0035060220,LA GLORIA [35060220],Pluviométrica,Convencional,Activa,Cundinamarca,Ubalá,"{'latitude': '-73.41977778', 'longitude': '4.8...",1845,-73.41977778,4.815694444,15/09/1964,Area Operativa 11 - Cundinamarca-Amazonas,Orinoco,Meta,Río Guavio,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,NaN,NaN


In [2]:
# Buscar la estación TIBAITATA [21205420]
# Se usa UTF-8 al guardar el CSV para que aparezcan bien los acentos

print(catalogoEstaciones.columns.tolist())

# Asegurar que la columna 'nombre' exista
if "nombre" not in catalogoEstaciones.columns:
    raise ValueError("No existe la columna 'nombre' en el DataFrame.")

# Filtro por nombre
# Este filtro se hace sobre texto normalizado en minúsculas para evitar errores con mayúsculas/acentos
catalogoEstaciones["nombre_norm"] = catalogoEstaciones["nombre"].astype(str).str.lower()

tibaitata_df = catalogoEstaciones[
    catalogoEstaciones["nombre_norm"].str.contains("tibaitata|tibaitatá", case=False, na=False, regex=True)
].copy()

# Filtro más preciso por código si existe
if "codigo" in catalogoEstaciones.columns:
    tibaitata_df = catalogoEstaciones[
        catalogoEstaciones["codigo"].astype(str)
        .str.contains("21205420", case=False, na=False, regex=False)
    ].copy()

# Mantener solo la fila final
if not tibaitata_df.empty:
    tibaitata_df = tibaitata_df.head(1).copy()

# Ver la fila final
display(tibaitata_df)

# Guardar con UTF-8 para conservar acentos y caracteres especiales
# La opción utf-8-sig ayuda también a Excel en Windows
if not tibaitata_df.empty:
    tibaitata_df.to_csv("./Datasets/tibaitata.csv", index=False, encoding="utf-8-sig")
else:
    print("No se encontró la estación TIBAITATA [21205420].")


['codigo', 'nombre', 'categoria', 'tecnologia', 'estado', 'departamento', 'municipio', 'ubicaci_n', 'altitud', 'longitud', 'latitud', 'fecha_instalacion', 'area_operativa', 'area_hidrografica', 'zona_hidrografica', 'subzona_hidrografica', 'entidad', 'corriente', 'fecha_suspension']


,codigo,nombre,categoria,tecnologia,estado,departamento,municipio,ubicaci_n,altitud,longitud,latitud,fecha_instalacion,area_operativa,area_hidrografica,zona_hidrografica,subzona_hidrografica,entidad,corriente,fecha_suspension,nombre_norm
3208,0021205420,TIBAITATA [21205420],Agrometeorológica,"Automática con Telemetría, Convencional",Activa,Cundinamarca,Mosquera,"{'latitude': '-74.205626', 'longitude': '4.688...",2543,-74.205626,4.688662,15/03/1954,Area Operativa 11 - Cundinamarca-Amazonas,Magdalena Cauca,Alto Magdalena,Río Bogotá,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,NaN,NaN,tibaitata [21205420]


## Radiación solar horaria: pasar a serie diaria

In [3]:

# Agrupar radiación solar horaria a diaria
actual_solar_files = [
    Path("./Datasets/Datos Tibaitata/Radiacion solar horaria/RadSolarHorarioTibaitata2005-2012.csv"),
    Path("./Datasets/Datos Tibaitata/Radiacion solar horaria/RadSolarHorariaTibaitata2012-2018.csv"),
    Path("./Datasets/Datos Tibaitata/Radiacion solar horaria/RadSolarHorarioTibaitata2018-2026.csv"),
]

frames_rad = []
for file in actual_solar_files:
    df = pd.read_csv(file)
    df.columns = [str(col).strip() for col in df.columns]
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df["Valor"] = pd.to_numeric(df["Valor"], errors="coerce")
    df = df.dropna(subset=["Fecha", "Valor"]).copy()
    df["Fecha"] = df["Fecha"].dt.normalize()
    df["NivelAprobacion"] = "Calculado"
    df["CodigoEstacion"] = df["CodigoEstacion"].astype(int)
    df["NombreEstacion"] = df["NombreEstacion"].astype(str).str.strip()
    df["Variable"] = df["Variable"].astype(str).str.strip()
    df["Parametro"] = df["Parametro"].astype(str).str.strip()
    df["Unidad"] = df["Unidad"].astype(str).str.strip()

    df_diario = (
        df.groupby(
            ["CodigoEstacion", "NombreEstacion", "Variable", "Parametro", "Unidad", "Fecha"],
            as_index=False,
        )
        .agg(Valor=("Valor", "sum"))
    )
    df_diario["NivelAprobacion"] = "Calculado"
    frames_rad.append(df_diario)

# Unificar en una sola serie diaria de radiación
df_rad_diaria = pd.concat(frames_rad, ignore_index=True).sort_values("Fecha").reset_index(drop=True)

print("Radiación solar diaria consolidada:")
print(df_rad_diaria.head(10).to_string(index=False))
print(f"\nNúmero de filas: {len(df_rad_diaria)}")
print(f"Rango de fechas: {df_rad_diaria['Fecha'].min()} a {df_rad_diaria['Fecha'].max()}")

# Guardar la serie diaria generada para usarla luego en la base consolidada
output_path = Path("./Datasets/Datos Tibaitata/RadSolarDiariaTibaitata.csv")
df_rad_diaria.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\nArchivo guardado: {output_path}")


Radiación solar diaria consolidada:
 CodigoEstacion             NombreEstacion  Variable                               Parametro Unidad      Fecha  Valor NivelAprobacion
       21206990 TIBAITATA - AUT [21206990] RAD SOLAR Radiación solar global horaria VALIDADA Wh/m^2 2005-02-02 1696.5       Calculado
       21206990 TIBAITATA - AUT [21206990] RAD SOLAR Radiación solar global horaria VALIDADA Wh/m^2 2005-02-03 4480.5       Calculado
       21206990 TIBAITATA - AUT [21206990] RAD SOLAR Radiación solar global horaria VALIDADA Wh/m^2 2005-02-04 2363.3       Calculado
       21206990 TIBAITATA - AUT [21206990] RAD SOLAR Radiación solar global horaria VALIDADA Wh/m^2 2005-02-05 3137.0       Calculado
       21206990 TIBAITATA - AUT [21206990] RAD SOLAR Radiación solar global horaria VALIDADA Wh/m^2 2005-02-06 1992.7       Calculado
       21206990 TIBAITATA - AUT [21206990] RAD SOLAR Radiación solar global horaria VALIDADA Wh/m^2 2005-02-07 3453.2       Calculado
       21206990 TIBAITATA 

## Datos para evapotranspiracion

### Inspección inicial de cada CSV



In [4]:
base_path = Path("./Datasets/Datos Tibaitata")
csv_files = sorted(base_path.glob("*.csv"))

print(f"Se encontraron {len(csv_files)} archivos CSV")
for file in csv_files:
    df = pd.read_csv(file)
    print(f"\n=== {file.name} ===")
    print(f"shape: {df.shape}")
    print(df.info())
    print(df.head(2).to_string(index=False))


Se encontraron 5 archivos CSV

=== HumedadRelativaTibaitata .csv ===
shape: (5531, 8)
<class 'pandas.DataFrame'>
RangeIndex: 5531 entries, 0 to 5530
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CodigoEstacion   5531 non-null   int64  
 1   NombreEstacion   5531 non-null   str    
 2   Variable         5531 non-null   str    
 3   Parametro        5531 non-null   str    
 4   Fecha            5531 non-null   str    
 5   Unidad           5531 non-null   str    
 6   Valor            5531 non-null   float64
 7   NivelAprobacion  5531 non-null   str    
dtypes: float64(1), int64(1), str(6)
memory usage: 345.8 KB
None
 CodigoEstacion             NombreEstacion     Variable                                         Parametro            Fecha Unidad     Valor NivelAprobacion
       21206990 TIBAITATA - AUT [21206990] HUM RELATIVA Humedad relativa del aire a 2 metros media diaria 2005-02-02 00:00      % 89.53125


=== TemperaturaMaximaTibaitata.csv ===
shape: (7027, 8)
<class 'pandas.DataFrame'>
RangeIndex: 7027 entries, 0 to 7026
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CodigoEstacion   7027 non-null   int64  
 1   NombreEstacion   7027 non-null   str    
 2   Variable         7027 non-null   str    
 3   Parametro        7027 non-null   str    
 4   Fecha            7027 non-null   str    
 5   Unidad           7027 non-null   str    
 6   Valor            7027 non-null   float64
 7   NivelAprobacion  7027 non-null   str    
dtypes: float64(1), int64(1), str(6)
memory usage: 439.3 KB
None
 CodigoEstacion       NombreEstacion    Variable                 Parametro            Fecha Unidad  Valor NivelAprobacion
       21205420 TIBAITATA [21205420] TEMPERATURA Temperatura máxima diaria 2005-01-03 00:00   degC   19.2      Definitivo
       21205420 TIBAITATA [21205420] TEMPERATURA Temperatura máxima diaria 2005

### Normalización y consolidación

In [20]:
base_path = Path("./Datasets/Datos Tibaitata")

def normalizar_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df["Valor"] = pd.to_numeric(df["Valor"], errors="coerce")
    df["Unidad"] = df["Unidad"].astype(str).str.strip()
    df["NombreEstacion"] = df["NombreEstacion"].astype(str).str.strip()
    df["Variable"] = df["Variable"].astype(str).str.strip()
    df["Parametro"] = df["Parametro"].astype(str).str.strip()
    return df.dropna(subset=["Fecha", "Valor"]).sort_values("Fecha").reset_index(drop=True)

# Cargar todos los CSV y normalizarlos
frames = {}
for csv_path in sorted(base_path.glob("*.csv")):
    df = pd.read_csv(csv_path)
    frames[csv_path.name] = normalizar_df(df)
    print(f"{csv_path.name}: {frames[csv_path.name].shape[0]} filas, fechas desde {frames[csv_path.name]['Fecha'].min()} hasta {frames[csv_path.name]['Fecha'].max()}")

# Concatenar en un solo DataFrame para análisis integrado
df_tibaitata = pd.concat(frames.values(), ignore_index=True)
df_tibaitata = df_tibaitata.sort_values(["Fecha", "Variable"]).reset_index(drop=True)

print("\nDataFrame consolidado:")
print(df_tibaitata.info())
print(df_tibaitata.head(3).to_string(index=False))

# Unidades estandarizadas (solo para visualización y no para perder la original)
unit_map = {
    "degC": "°C",
    "Wh/m^2": "Wh/m²",
    "m/s": "m/s",
    "%": "%",
}
df_tibaitata["Unidad_esta"] = df_tibaitata["Unidad"].map(unit_map).fillna(df_tibaitata["Unidad"] )

print("\nUnidades presentes:")
print(df_tibaitata["Unidad_esta"].value_counts())

# Copia inmutable de referencia para comparaciones ANTES/DESPUÉS
df_tibaitata_raw = df_tibaitata.copy()
df_tibaitata.to_pickle("./Datasets/ConsolidadoTibaitata.pkl")


HumedadRelativaTibaitata .csv: 5531 filas, fechas desde 2005-02-02 00:00:00 hasta 2023-03-16 00:00:00
RadSolarDiariaTibaitata.csv: 6029 filas, fechas desde 2005-02-02 00:00:00 hasta 2022-10-31 00:00:00
TemperaturaMaximaTibaitata.csv: 7027 filas, fechas desde 2005-01-03 00:00:00 hasta 2026-01-01 00:00:00
TemperaturaMinimaTibaitata.csv: 14307 filas, fechas desde 2005-01-01 00:00:00 hasta 2026-01-01 00:00:00
VelViento10MinDiariaTibaitata.csv: 3497 filas, fechas desde 2005-02-02 00:00:00 hasta 2023-03-16 00:00:00

DataFrame consolidado:
<class 'pandas.DataFrame'>
RangeIndex: 36391 entries, 0 to 36390
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CodigoEstacion   36391 non-null  int64         
 1   NombreEstacion   36391 non-null  str           
 2   Variable         36391 non-null  str           
 3   Parametro        36391 non-null  str           
 4   Fecha            36391 non-null  datetime

In [18]:
df_tibaitata.dtypes

CodigoEstacion              int64
NombreEstacion                str
Variable                      str
Parametro                     str
Fecha              datetime64[us]
Unidad                        str
Valor                     float64
NivelAprobacion               str
Unidad_esta                   str
dtype: object